In [1]:
from ultralytics import YOLO

# Load YOLO26s pretrained weights
model = YOLO('yolo26s.pt')

# Phase 1 - Initial training
results = model.train(
    data    = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    project = r"C:\Users\Christian\AvianTech\models\yolo26",
    name    = "aviantech_v26_phase1",
    pretrained = True,
    optimizer  = "auto",   # Let YOLO26 use its MuSGD optimizer
)

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Christian\AvianTech\config\birds.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aviantech_v26_phase1, nbs=64, nms=False, opset=None, optimize=False, optimize

In [1]:
# First, verify CUDA is working
import torch
print(torch.cuda.is_available())        # Must print True
print(torch.cuda.get_device_name(0))    # Must print RTX 4050

True
NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
from ultralytics import YOLO

best_weights = r"C:\Users\Christian\AvianTech\models\yolo26\aviantech_v26_phase1\weights\best.pt"
model = YOLO(best_weights)

results = model.train(
    data      = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    device    = 0,
    project   = r"C:\Users\Christian\AvianTech\models\yolo26",
    name      = "aviantech_v26_phase2",
    optimizer = "auto",
    workers   = 0,
    exist_ok  = True,
    patience  = 20,
)

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Christian\AvianTech\config\birds.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\Christian\AvianTech\models\yolo26\aviantech_v26_phase1\weights\best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aviantec

In [3]:
from ultralytics import YOLO

# Use Phase 1 best — it outperformed Phase 2
model = YOLO(r"C:\Users\Christian\AvianTech\models\yolo26\aviantech_v26_phase1\weights\best.pt")

metrics = model.val(
    data    = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    split   = "test",
    device  = 0,
    verbose = True,
    plots   = True,
)

print(f"mAP@50     : {metrics.box.map50:.4f}")
print(f"mAP@50-95  : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO26s summary (fused): 122 layers, 9,479,499 parameters, 0 gradients, 20.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 171.182.1 MB/s, size: 44.0 KB)
val: Scanning C:\Users\Christian\AvianTech\data\datasets\test\labels.cache... 185 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.8it/s 3.2s0.2s
                   all        185        290      0.805      0.721      0.787      0.703
   Amethyst brown dove          5          8      0.645       0.75       0.72      0.614
 Asian Glossy Starling          5          8      0.753      0.875      0.853      0.818
Blue-capped Kingfisher          9         13      0.906      0.846      0.871      0.799
Blue-crowned Racquet-tail          4          5      0.637        0.6      0.616      0.549
    

In [4]:
import pandas as pd

# Reuse BirdNET results from YOLOv11 run — audio hasn't changed
xlsx_path = r"C:\Users\Christian\AvianTech\results_charts_v11\aviantech_full_results.xlsx"
df_audio  = pd.read_excel(xlsx_path, sheet_name="BirdNET Results")

print(f"✅ Loaded BirdNET results: {len(df_audio)} species")
print(df_audio[["species", "detected", "confidence", "match"]].to_string(index=False))

✅ Loaded BirdNET results: 37 species
                                           species                  detected  confidence match
      Golden-bellied Gerygone (Gerygone sulphurea)   Golden-bellied Gerygone    0.998742     ✅
               Garden Sunbird (Cinnyris jugularis)      Olive-backed Sunbird    0.997519    ⚠️
  Philippine Pied-Fantail (Rhipidura nigritorquis)   Philippine Pied-Fantail    0.976833     ✅
              Philippine Trogon (Harpactes ardens)         Philippine Trogon    0.974687     ✅
              Philippine Coucal(Centropus viridis)         Philippine Coucal    0.974051     ✅
    Yellow-wattled Bulbul (Microtarsus urostictus)     Yellow-wattled Bulbul    0.936647     ✅
 Striated Wren (Babbler - Ptilocichla mindanensis)     Striated Wren-Babbler    0.902584    ⚠️
                       Rock Pigeon (Columba livia)               Rock Pigeon    0.899085     ✅
        Red-keeled Flowerpecker (Dicaeum australe)   Red-keeled Flowerpecker    0.891633     ✅
             

In [5]:
from ultralytics import YOLO
import pandas as pd

model = YOLO(r"C:\Users\Christian\AvianTech\models\yolo26\aviantech_v26_phase1\weights\best.pt")

metrics = model.val(
    data    = r"C:\Users\Christian\AvianTech\config\birds.yaml",
    split   = "test",
    device  = 0,
    verbose = False,
)

class_names = list(model.names.values())
ap50_scores = metrics.box.ap50

df_yolo26 = pd.DataFrame({
    "species"   : class_names,
    "yolo_ap50" : ap50_scores,
})

print(f"✅ YOLO26 per-class AP50 loaded: {len(df_yolo26)} species")

Ultralytics 8.4.21  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO26s summary (fused): 122 layers, 9,479,499 parameters, 0 gradients, 20.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 279.1150.8 MB/s, size: 38.4 KB)
val: Scanning C:\Users\Christian\AvianTech\data\datasets\test\labels.cache... 185 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 5.9it/s 2.0s0.2s
                   all        185        290      0.805      0.721      0.787      0.703
Speed: 1.0ms preprocess, 6.7ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\Users\Christian\AvianTech\runs\detect\val9
✅ YOLO26 per-class AP50 loaded: 37 species


In [8]:
import pandas as pd
import numpy as np

# Reload BirdNET results
xlsx_path = r"C:\Users\Christian\AvianTech\results_charts_v11\aviantech_full_results.xlsx"
df_audio  = pd.read_excel(xlsx_path, sheet_name="BirdNET Results")

# Fix: strip everything in parentheses from audio species names
df_audio["key"] = df_audio["species"].str.replace(r"\s*\(.*\)", "", regex=True).str.lower().str.strip()
df_yolo26["key"] = df_yolo26["species"].str.lower().str.strip()

# Preview the keys to confirm they match
print("🎵 BirdNET keys sample:")
print(df_audio["key"].head(10).to_string())
print("\n👁️ YOLO26 keys sample:")
print(df_yolo26["key"].head(10).to_string())

# Merge
df_fused = pd.merge(df_yolo26, df_audio[["key", "confidence", "match"]],
                    on="key", how="left")
df_fused = df_fused.rename(columns={"confidence": "birdnet_conf"})
df_fused["birdnet_conf"] = df_fused["birdnet_conf"].fillna(0.0)

# Fusion: 60% YOLO26 + 40% BirdNET
df_fused["fusion_score"] = (
    df_fused["yolo_ap50"]    * 0.6 +
    df_fused["birdnet_conf"] * 0.4
)

df_fused = df_fused.sort_values("fusion_score", ascending=False).reset_index(drop=True)
df_fused.index += 1

print("\n🔗 AvianTech — Fused Detection Results (YOLO26 + BirdNET)\n")
print(f"{'#':<4} {'Species':<38} {'YOLO AP50':>10} {'BirdNET':>9} {'Fusion':>9}  {'Audio'}")
print("─" * 80)
for idx, row in df_fused.iterrows():
    print(f"  {idx:<3} {row['species'][:37]:<38} {row['yolo_ap50']:>9.1%} "
          f"{row['birdnet_conf']:>8.1%} {row['fusion_score']:>8.1%}  {row.get('match', 'N/A')}")

print(f"\n📊 Average Fusion Score : {df_fused['fusion_score'].mean():.2%}")
print(f"📊 Average YOLO AP50   : {df_fused['yolo_ap50'].mean():.2%}")
print(f"📊 Average BirdNET Conf: {df_fused['birdnet_conf'].mean():.2%}")

🎵 BirdNET keys sample:
0    golden-bellied gerygone
1             garden sunbird
2    philippine pied-fantail
3          philippine trogon
4          philippine coucal
5      yellow-wattled bulbul
6              striated wren
7                rock pigeon
8    red-keeled flowerpecker
9      green imperial pigeon

👁️ YOLO26 keys sample:
0          amethyst brown dove
1        asian glossy starling
2       blue-capped kingfisher
3    blue-crowned racquet-tail
4            blue-naped parrot
5                 brown shrike
6            celestial monarch
7               chestnut munia
8                       coleto
9        eurasian tree sparrow

🔗 AvianTech — Fused Detection Results (YOLO26 + BirdNET)

#    Species                                 YOLO AP50   BirdNET    Fusion  Audio
────────────────────────────────────────────────────────────────────────────────
  1   Philippine Trogon                          97.9%    97.5%    97.7%  ✅
  2   Striated Wren                              99.5% 

In [9]:
import os

save_dir  = r"C:\Users\Christian\AvianTech\results_charts_v26"
os.makedirs(save_dir, exist_ok=True)
xlsx_path = os.path.join(save_dir, "aviantech_v26_full_results.xlsx")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_fused.drop(columns=["key"]).to_excel(
        writer, sheet_name="Fusion Results", index=True)
    df_audio[["species", "detected", "confidence", "match"]].to_excel(
        writer, sheet_name="BirdNET Results", index=False)
    df_yolo26[["species", "yolo_ap50"]].to_excel(
        writer, sheet_name="YOLO26 Per-Class AP50", index=False)

print(f"✅ Results saved → {xlsx_path}")

✅ Results saved → C:\Users\Christian\AvianTech\results_charts_v26\aviantech_v26_full_results.xlsx


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Paths ──────────────────────────────────────────────────────────────────
results_csv = r"C:\Users\Christian\AvianTech\models\yolo26\aviantech_v26_phase1\results.csv"
runs_dir    = r"C:\Users\Christian\AvianTech\models\yolo26\aviantech_v26_phase1"
dataset_dir = r"C:\Users\Christian\AvianTech\data\datasets"
save_dir    = r"C:\Users\Christian\AvianTech\results_charts_v26"
os.makedirs(save_dir, exist_ok=True)

# ── Load CSV ────────────────────────────────────────────────────────────────
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
epochs = df["epoch"]

# ── Style ───────────────────────────────────────────────────────────────────
BG, CARD = "#0f1117", "#1a1d27"
GRN, ACC, YLW, PRP, RED = "#00ff88", "#00d4ff", "#ffa502", "#a29bfe", "#ff4757"
WHT, GRY = "#ffffff", "#8892a4"

best_idx = df["metrics/mAP50(B)"].idxmax()
best     = df.loc[best_idx]

def base_fig(title):
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_edgecolor("#2a2d3a")
    ax.tick_params(colors=GRY, labelsize=9)
    ax.set_title(title, color=WHT, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Epoch", color=GRY, fontsize=10)
    ax.grid(color="#2a2d3a", linewidth=0.5)
    ax.axvline(x=best["epoch"], color=YLW, linestyle="--",
               linewidth=1.2, alpha=0.7, label=f"Best epoch ({int(best['epoch'])})")
    return fig, ax

def save(fig, name):
    path = os.path.join(save_dir, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show(); plt.close(fig)
    print(f"✅ Saved → {path}")

# ════════════════════════════════════════════════════════════════════════════
# 1. mAP Curves
# ════════════════════════════════════════════════════════════════════════════
fig, ax = base_fig("mAP Curves — YOLO26s")
ax.plot(epochs, df["metrics/mAP50(B)"],    color=GRN, linewidth=2.5, label=f"mAP@0.5  (best={best['metrics/mAP50(B)']:.4f})")
ax.plot(epochs, df["metrics/mAP50-95(B)"], color=ACC, linewidth=2.5, label=f"mAP@0.5:0.95  (best={best['metrics/mAP50-95(B)']:.4f})")
ax.fill_between(epochs, df["metrics/mAP50(B)"],    alpha=0.12, color=GRN)
ax.fill_between(epochs, df["metrics/mAP50-95(B)"], alpha=0.12, color=ACC)
ax.set_ylabel("mAP", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "1_mAP_curves.png")

# ════════════════════════════════════════════════════════════════════════════
# 2. Precision & Recall
# ════════════════════════════════════════════════════════════════════════════
fig, ax = base_fig("Precision & Recall — YOLO26s")
ax.plot(epochs, df["metrics/precision(B)"], color=YLW, linewidth=2.5, label=f"Precision  (best={best['metrics/precision(B)']:.4f})")
ax.plot(epochs, df["metrics/recall(B)"],    color=PRP, linewidth=2.5, label=f"Recall  (best={best['metrics/recall(B)']:.4f})")
ax.fill_between(epochs, df["metrics/precision(B)"], alpha=0.12, color=YLW)
ax.fill_between(epochs, df["metrics/recall(B)"],    alpha=0.12, color=PRP)
ax.set_ylabel("Score", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "2_precision_recall.png")

# ════════════════════════════════════════════════════════════════════════════
# 3. Training Losses
# ════════════════════════════════════════════════════════════════════════════
fig, ax = base_fig("Training Losses — YOLO26s")
ax.plot(epochs, df["train/box_loss"], color=RED, linewidth=2.5, label="Box Loss")
ax.plot(epochs, df["train/cls_loss"], color=PRP, linewidth=2.5, label="Class Loss")
ax.plot(epochs, df["train/dfl_loss"], color=YLW, linewidth=2.5, label="DFL Loss")
ax.fill_between(epochs, df["train/box_loss"], alpha=0.10, color=RED)
ax.fill_between(epochs, df["train/cls_loss"], alpha=0.10, color=PRP)
ax.fill_between(epochs, df["train/dfl_loss"], alpha=0.10, color=YLW)
ax.set_ylabel("Loss", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "3_train_losses.png")

# ════════════════════════════════════════════════════════════════════════════
# 4. Validation Losses
# ════════════════════════════════════════════════════════════════════════════
fig, ax = base_fig("Validation Losses — YOLO26s")
ax.plot(epochs, df["val/box_loss"], color=RED, linewidth=2.5, label="Val Box Loss")
ax.plot(epochs, df["val/cls_loss"], color=PRP, linewidth=2.5, label="Val Class Loss")
ax.plot(epochs, df["val/dfl_loss"], color=YLW, linewidth=2.5, label="Val DFL Loss")
ax.fill_between(epochs, df["val/box_loss"], alpha=0.10, color=RED)
ax.fill_between(epochs, df["val/cls_loss"], alpha=0.10, color=PRP)
ax.fill_between(epochs, df["val/dfl_loss"], alpha=0.10, color=YLW)
ax.set_ylabel("Loss", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "4_val_losses.png")

# ════════════════════════════════════════════════════════════════════════════
# 5. F1 Score
# ════════════════════════════════════════════════════════════════════════════
p  = df["metrics/precision(B)"]
r  = df["metrics/recall(B)"]
f1 = 2 * p * r / (p + r + 1e-8)
fig, ax = base_fig("F1 Score — YOLO26s")
ax.plot(epochs, f1, color=RED, linewidth=2.5, label=f"F1  (best={f1.max():.4f})")
ax.fill_between(epochs, f1, alpha=0.12, color=RED)
ax.set_ylabel("F1 Score", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "5_f1_score.png")

# ════════════════════════════════════════════════════════════════════════════
# 6. Box Loss Train vs Val
# ════════════════════════════════════════════════════════════════════════════
fig, ax = base_fig("Box Loss — Train vs Validation — YOLO26s")
ax.plot(epochs, df["train/box_loss"], color=RED, linewidth=2.5, label="Train Box Loss")
ax.plot(epochs, df["val/box_loss"],   color=ACC, linewidth=2.5, linestyle="--", label="Val Box Loss")
ax.set_ylabel("Loss", color=GRY, fontsize=10)
ax.legend(facecolor=CARD, edgecolor=GRY, labelcolor=WHT, fontsize=9)
save(fig, "6_box_loss_comparison.png")

# ════════════════════════════════════════════════════════════════════════════
# 7. Dataset Split Donut
# ════════════════════════════════════════════════════════════════════════════
splits = {}
for split in ["train", "val", "test"]:
    img_path = os.path.join(dataset_dir, split, "images")
    splits[split] = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
total     = sum(splits.values())
split_pct = {k: v / total * 100 for k, v in splits.items()}

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
sizes  = [splits["train"], splits["val"], splits["test"]]
colors = [GRN, ACC, YLW]
labels = [
    f"Train — {splits['train']} images ({split_pct['train']:.0f}%)",
    f"Val   — {splits['val']} images ({split_pct['val']:.0f}%)",
    f"Test  — {splits['test']} images ({split_pct['test']:.0f}%)",
]
wedges, _ = ax.pie(sizes, colors=colors, startangle=90,
                   wedgeprops=dict(width=0.52, edgecolor=BG, linewidth=3))
ax.legend(wedges, labels, loc="lower center", facecolor=CARD,
          edgecolor=GRY, labelcolor=WHT, fontsize=11,
          bbox_to_anchor=(0.5, -0.08))
ax.text(0, 0.08, f"{total}", ha="center", va="center",
        fontsize=28, fontweight="bold", color=WHT)
ax.text(0, -0.18, "total images", ha="center", va="center",
        fontsize=12, color=GRY)
ax.set_title("Dataset Split  (70 / 20 / 10)", color=WHT,
             fontsize=14, fontweight="bold", pad=20)
save(fig, "7_dataset_split.png")

# ════════════════════════════════════════════════════════════════════════════
# 8. Confusion Matrix
# ════════════════════════════════════════════════════════════════════════════
cm_path = os.path.join(runs_dir, "confusion_matrix.png")
if os.path.exists(cm_path):
    fig, ax = plt.subplots(figsize=(14, 12))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    cm_img = plt.imread(cm_path)
    ax.imshow(cm_img)
    ax.set_title("Confusion Matrix — 37 Philippine Bird Species — YOLO26s",
                 color=WHT, fontsize=14, fontweight="bold", pad=12)
    ax.axis("off")
    save(fig, "8_confusion_matrix.png")
else:
    print("⚠️  confusion_matrix.png not found in:", runs_dir)

# ════════════════════════════════════════════════════════════════════════════
# 9. KPI Summary Card
# ════════════════════════════════════════════════════════════════════════════
best_f1 = f1.max()

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_edgecolor(ACC); sp.set_linewidth(2)
ax.axis("off")

kpis = [
    ("mAP@0.5",      f"{best['metrics/mAP50(B)']*100:.2f}%",    GRN),
    ("mAP@0.5:0.95", f"{best['metrics/mAP50-95(B)']*100:.2f}%", ACC),
    ("Precision",    f"{best['metrics/precision(B)']*100:.2f}%", YLW),
    ("Recall",       f"{best['metrics/recall(B)']*100:.2f}%",    PRP),
    ("F1 Score",     f"{best_f1*100:.2f}%",                      RED),
    ("Best Epoch",   f"{int(best['epoch'])} / {int(df['epoch'].iloc[-1])}", WHT),
]

for i, (label, val, color) in enumerate(kpis):
    x = 0.08 + i * 0.155
    ax.text(x, 0.72, val,   transform=ax.transAxes, ha="center",
            fontsize=20, fontweight="bold", color=color)
    ax.text(x, 0.42, label, transform=ax.transAxes, ha="center",
            fontsize=10, color=WHT, fontweight="bold")

ax.text(0.5, 0.12,
        f"Dataset: {total} images  |  37 Species  |  Split: {splits['train']}/{splits['val']}/{splits['test']}  (70/20/10)  |  YOLO26s  |  {int(df['epoch'].iloc[-1])} Epochs",
        transform=ax.transAxes, ha="center", fontsize=9, color=GRY)

ax.set_title("AvianTech — YOLO26s Final Model Performance Summary",
             color=WHT, fontsize=14, fontweight="bold", pad=14)
save(fig, "9_kpi_summary.png")

print(f"\n🎉 All 9 charts saved to:\n   {save_dir}")

<Figure size 1000x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v26\1_mAP_curves.png


<Figure size 1000x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v26\2_precision_recall.png


<Figure size 1000x500 with 1 Axes>

...

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v26\7_dataset_split.png


<Figure size 1400x1200 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v26\8_confusion_matrix.png


<Figure size 1200x500 with 1 Axes>

✅ Saved → C:\Users\Christian\AvianTech\results_charts_v26\9_kpi_summary.png

🎉 All 9 charts saved to:
   C:\Users\Christian\AvianTech\results_charts_v26
